# Research-Grade Handwritten OCR Reconstruction Pipeline
## For Google Colab • Confidence-aware • RAG-assisted • Noise-resilient

**Purpose**: Extract handwritten student answers from scanned sheets while:
- ✓ Preserving student mistakes (NOT correcting)
- ✓ Recovering OCR corruption
- ✓ Using confidence-aware reconstruction
- ✓ Leveraging RAG context (open-source)
- ✓ Producing **two .txt files**: `original_ocr.txt` + `reconstructed.txt`
- ✓ RAG topics: LAN / Wired / Wireless / Client-Server networking

**Key Principle**: This is OCR noise recovery, NOT grammar correction or answer generation.

## Cell 1: Install Dependencies

In [ ]:
# Install required packages (bitsandbytes omitted — often breaks CPU-only Colab)
!pip install -q kraken
!pip install -q torch torchvision
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q opencv-python pillow
!pip install -q numpy scipy scikit-learn
!pip install -q tqdm
!pip install -q accelerate

print("✓ All dependencies installed")

## Cell 2: Import Modules & Setup

In [ ]:
import sys
import json
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import torch

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB")

print(f"PyTorch version: {torch.__version__}")

## Cell 3: Load OCR Pipeline Module

Copy the pipeline code into this cell or upload the Python file.

In [ ]:
import os
import sys
from pathlib import Path

PIPELINE_MODULE = "ocr_reconstruction_pipeline.py"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Running in Google Colab")
    print("Upload ocr_reconstruction_pipeline.py when prompted...")
    uploaded = files.upload()
    if PIPELINE_MODULE not in uploaded:
        raise FileNotFoundError(
            f"Please upload {PIPELINE_MODULE} (the pipeline .py file from your machine)."
        )
    module_path = Path("/content") / PIPELINE_MODULE
else:
    print("Running locally")
    # Same folder as this notebook, or current working directory
    candidates = [
        Path.cwd() / PIPELINE_MODULE,
        Path(r"c:\Users\User\Downloads") / PIPELINE_MODULE,
    ]
    module_path = next((p for p in candidates if p.is_file()), None)
    if module_path is None:
        raise FileNotFoundError(
            f"Place {PIPELINE_MODULE} next to this notebook or in the working directory."
        )

sys.path.insert(0, str(module_path.parent))
print(f"✓ Pipeline module path: {module_path}")

## Cell 4: Configure Pipeline

In [ ]:
# Import after module is loaded
from ocr_reconstruction_pipeline import (
    PipelineConfig,
    OCRReconstructionPipeline,
    PipelineOutput,
    create_sample_image
)

# Create configuration (customize as needed)
# PipelineConfig auto-falls back to CPU and FP32 when CUDA is unavailable.
config = PipelineConfig(
    device="cuda",
    dtype="fp16",
    batch_size=4,
    max_workers=4,
    
    # OCR models
    kraken_model="blla.mlmodel",
    ocr_model="microsoft/trocr-base-handwritten",
    
    # Confidence thresholds
    high_confidence_threshold=0.85,
    medium_confidence_threshold=0.60,
    
    # RAG
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    rag_top_k=3,
    similarity_threshold=0.5,
    
    # Reconstruction: use "rag" (recommended) — LLM was echoing the prompt
    reconstruction_method="rag",
    reconstruction_model="google/flan-t5-base",
    reconstruction_dtype="fp16",
    max_reconstruction_length=512,
    
    # Logging
    verbose=True,
    save_intermediate=False,
)

print("Configuration:")
print(json.dumps(config.to_dict(), indent=2))

## Cell 5: Initialize Pipeline (Downloads Models)

⚠️ **WARNING**: This will download ~8-10GB of models. Requires good internet and free Colab storage.

First time only.

In [ ]:
# Initialize the complete pipeline
# This downloads: TrOCR, Sentence Transformers, FLAN-T5

print("Initializing OCR Reconstruction Pipeline...")
print("This may take 5-10 minutes on first run (model downloads)\n")

try:
    pipeline = OCRReconstructionPipeline(config)
    print("\n✓ Pipeline initialized successfully!")
except Exception as e:
    print(f"✗ Error initializing pipeline: {e}")
    import traceback
    traceback.print_exc()

## Cell 6: Test with Sample Image

In [ ]:
# Create sample test image
print("Creating sample handwritten image for testing...")
sample_image = create_sample_image()

print(f"Sample image shape: {sample_image.shape}")
print(f"Sample image dtype: {sample_image.dtype}")

# Display preview
try:
    from IPython.display import Image as IPImage, display
    import tempfile
    preview_path = os.path.join(tempfile.gettempdir(), "ocr_sample_preview.png")
    cv2.imwrite(preview_path, sample_image)
    display(IPImage(preview_path))
except Exception as exc:
    print(f"Could not display image: {exc}")

## Cell 7: Run Full Pipeline on Sample Image

In [ ]:
# Process sample image through complete pipeline
print("Processing sample image...\n")

OUTPUT_DIR = "/content/ocr_output" if IN_COLAB else str(Path.cwd() / "ocr_output")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

try:
    output = pipeline.process_image(sample_image, text_output_dir=OUTPUT_DIR)
    raw_txt, recon_txt = output.save_text_outputs(OUTPUT_DIR)

    print("\n" + "="*80)
    print("PIPELINE OUTPUT")
    print("="*80)

    print(f"\nRaw OCR:\n{output.raw_ocr}")
    print(f"\nReconstructed:\n{output.reconstructed_text}")
    print(f"\nDetected Topic: {output.topic}")
    print(f"\nText files saved:")
    print(f"  {raw_txt}")
    print(f"  {recon_txt}")
    print(f"\nConfidence Summary:")
    for key, val in output.confidence_summary.items():
        print(f"  {key}: {val}")

except Exception as e:
    print(f"✗ Error processing image: {e}")
    import traceback
    traceback.print_exc()

## Cell 8: View Full JSON Output

In [ ]:
# Display full output as formatted JSON
if 'output' in locals():
    json_str = output.to_json()
    print(json_str)
else:
    print("No output available. Run Cell 7 first.")

## Cell 9: Save Output (.txt + optional JSON)

In [ ]:
# Primary output: two plain-text files
if 'output' in locals():
    save_dir = OUTPUT_DIR if 'OUTPUT_DIR' in globals() else (
        "/content/ocr_output" if IN_COLAB else str(Path.cwd() / "ocr_output")
    )
    raw_txt, recon_txt = output.save_text_outputs(save_dir)
    print("✓ Saved text outputs:")
    print(f"  Original OCR:     {raw_txt}")
    print(f"  Reconstructed:    {recon_txt}")

    # Optional: full JSON for debugging/evaluation
    json_path = str(Path(save_dir) / "ocr_output.json")
    output.save(json_path)
    print(f"  (optional JSON): {json_path}")
else:
    print("No output available. Run Cell 7 first.")

## Cell 10: Upload & Process Your Own Image

Upload a scanned handwritten answer sheet.

In [ ]:
print("Upload a scanned handwritten answer sheet image...")

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    image_path = f"/content/{filename}"
else:
    image_path = input("Enter full path to your image file: ").strip().strip('"')
    filename = os.path.basename(image_path)

print(f"✓ Using: {filename}")

user_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
if user_image is None:
    user_image = cv2.imread(image_path)
    if user_image is not None and len(user_image.shape) == 3:
        user_image = cv2.cvtColor(user_image, cv2.COLOR_BGR2GRAY)

if user_image is None:
    raise ValueError(f"Could not read image: {image_path}")

print(f"Image shape: {user_image.shape}")
print(f"Image dtype: {user_image.dtype}")

## Cell 11: Process Your Image

In [ ]:
# Process your uploaded image
print("Processing your image...\n")

user_output_dir = "/content/user_ocr_output" if IN_COLAB else str(Path.cwd() / "user_ocr_output")
Path(user_output_dir).mkdir(parents=True, exist_ok=True)

try:
    user_output = pipeline.process_image(user_image, text_output_dir=user_output_dir)
    user_raw_txt, user_recon_txt = user_output.save_text_outputs(user_output_dir)

    print("\n" + "="*80)
    print("RESULTS FOR YOUR IMAGE")
    print("="*80)

    print(f"\nRaw OCR:\n{user_output.raw_ocr}")
    print(f"\n" + "-"*80)
    print(f"\nReconstructed:\n{user_output.reconstructed_text}")
    print(f"\nTopic: {user_output.topic}")
    print(f"\nText files saved:")
    print(f"  {user_raw_txt}")
    print(f"  {user_recon_txt}")
    
    # Show line-by-line results
    print(f"\n" + "-"*80)
    print("Line-by-line Confidence:")
    for i, line in enumerate(user_output.lines, 1):
        print(f"\nLine {i}: {line.text[:60]}...")
        print(f"  Confidence: {line.confidence:.4f}")
        print(f"  Low-conf words: {[w.word for w in line.words if w.is_low_confidence]}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

## Cell 12: Save Your Results

In [ ]:
# Download the two .txt result files (Colab)
if 'user_output' in locals():
    save_dir = user_output_dir if 'user_output_dir' in dir() else (
        "/content/user_ocr_output" if IN_COLAB else str(Path.cwd() / "user_ocr_output")
    )
    raw_txt, recon_txt = user_output.save_text_outputs(save_dir)
    print("✓ Text outputs:")
    print(f"  {raw_txt}")
    print(f"  {recon_txt}")

    if IN_COLAB:
        from google.colab import files
        files.download(raw_txt)
        files.download(recon_txt)
        print("✓ Downloaded original_ocr.txt and reconstructed.txt")
else:
    print("No results to save. Run Cell 11 first.")

## Cell 13: Advanced Configuration Examples

Customize pipeline behavior.

In [ ]:
# Example 1: Use smaller model for faster inference
config_fast = PipelineConfig(
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype="fp16",
    ocr_model="microsoft/trocr-base-handwritten",  # Smaller model
    reconstruction_model="google/flan-t5-base",     # Smaller model
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    verbose=True,
)

print("Fast Configuration (for T4 GPU with 16GB VRAM):")
print(json.dumps(config_fast.to_dict(), indent=2))

# Example 2: Use larger model for better quality
config_quality = PipelineConfig(
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype="fp16",
    ocr_model="microsoft/trocr-large-handwritten",  # Larger model
    reconstruction_model="google/flan-t5-xl",       # Larger model
    embedding_model="sentence-transformers/all-mpnet-base-v2",
    verbose=True,
)

print("\n\nQuality Configuration (requires higher VRAM):")
print(json.dumps(config_quality.to_dict(), indent=2))

## Cell 14: Manage RAG Topic Database

Add custom topics for your specific domain.

In [ ]:
# View built-in LAN RAG topics (add more with add_topic if needed)
if 'pipeline' in locals():
    print(f"Total topics in database: {len(pipeline.rag.topic_database)}")
    for i, doc in enumerate(pipeline.rag.topic_database, 1):
        print(f"  {i}. {doc.heading}")

    # Example: add another networking topic
    # pipeline.rag.add_topic(
    #     heading="Peer-to-Peer LAN",
    #     content="In peer-to-peer LAN, each computer can act as client and server...",
    #     keywords=["P2P", "peer-to-peer", "distributed"],
    # )

## Cell 15: Benchmark & Profile

In [ ]:
# Benchmark pipeline performance
import time

if 'pipeline' in locals():
    print("Benchmarking pipeline...\n")
    
    # Create test images of different sizes
    test_sizes = [(300, 600), (400, 800), (500, 1000)]
    
    results = []
    
    for height, width in test_sizes:
        test_image = np.ones((height, width), dtype=np.uint8) * 255
        
        # Add some text
        cv2.putText(test_image, "Test line 1", (30, 50),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, 0, 2)
        
        start = time.time()
        output = pipeline.process_image(test_image)
        elapsed = time.time() - start
        
        total_time = sum(output.timings_ms.values())
        results.append({
            "size": f"{width}x{height}",
            "total_ms": total_time,
            "throughput_ms": elapsed * 1000
        })
        
        print(f"Size {width}x{height}: {total_time:.2f}ms")
    
    print("\nBenchmark complete!")

## Cell 16: Memory Optimization

In [ ]:
# Monitor GPU memory
if torch.cuda.is_available():
    print("GPU Memory Status:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f}GB")
    print(f"  Cached: {torch.cuda.memory_reserved() / 1e9:.2f}GB")
    print(f"  Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB")
    
    # Clear cache if needed
    if torch.cuda.memory_allocated() > 10e9:
        print("\nClearing GPU cache...")
        torch.cuda.empty_cache()
        print(f"  Allocated after clear: {torch.cuda.memory_allocated() / 1e9:.2f}GB")
else:
    print("No CUDA GPU detected")

## Cell 17: FastAPI Server (Optional)

Deploy as REST API in Colab using Ngrok.

In [ ]:
# Optional: Create simple FastAPI wrapper

from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import uvicorn
from io import BytesIO

app = FastAPI(title="OCR Reconstruction API")

@app.post("/process")
async def process_image(file: UploadFile = File(...)):
    """Process uploaded image through OCR pipeline."""
    try:
        # Read image
        contents = await file.read()
        nparr = np.frombuffer(contents, np.uint8)
        image = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)
        
        if 'pipeline' not in globals():
            return JSONResponse(
                {"error": "Pipeline not initialized"},
                status_code=500
            )
        
        # Process
        output = pipeline.process_image(image)
        
        # Return JSON
        return JSONResponse(json.loads(output.to_json()))
    
    except Exception as e:
        return JSONResponse(
            {"error": str(e)},
            status_code=500
        )

@app.get("/health")
async def health():
    """Health check."""
    return {"status": "healthy"}

# To run server in Colab:
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.connect(8000)
# uvicorn.run(app, host="0.0.0.0", port=8000)

print("FastAPI app defined. To run:")
print("  !pip install -q pyngrok")
print("  from pyngrok import ngrok")
print("  public_url = ngrok.connect(8000)")
print("  uvicorn.run(app, host='0.0.0.0', port=8000)")

## Cell 18: Troubleshooting & Debugging

In [ ]:
# Common issues & solutions

print("="*80)
print("TROUBLESHOOTING GUIDE")
print("="*80)

print("\n1. Out of Memory (OOM) Error:")
print("   - Use FP16 dtype (already set)")
print("   - Reduce batch_size in config")
print("   - Use smaller models (trocr-base, flan-t5-base)")
print("   - Clear GPU cache: torch.cuda.empty_cache()")

print("\n2. Model Download Failures:")
print("   - Check internet connection")
print("   - Clear Hugging Face cache: rm -rf ~/.cache/huggingface")
print("   - Manually download: transformers-cli download model-name")

print("\n3. Kraken Segmentation Issues:")
print("   - Ensure image is grayscale uint8")
print("   - Try with downscaled image")
print("   - Check image quality (good contrast)")

print("\n4. Low OCR Accuracy:")
print("   - Preprocess image: enhance contrast, remove shadows")
print("   - Use larger TrOCR model (trocr-large-handwritten)")
print("   - Adjust segmentation parameters")

print("\n5. Reconstruction Quality:")
print("   - Check confidence scores")
print("   - Verify RAG context is relevant")
print("   - Use larger reconstruction model (FLAN-T5-XL)")

print("\n" + "="*80)

## Cell 19: References & Model Information

In [ ]:
# Model information & resources

print("="*80)
print("MODELS USED IN THIS PIPELINE")
print("="*80)

models_info = {
    "Segmentation": {
        "Model": "Kraken BLLA",
        "Purpose": "Baseline + Line Detection",
        "Input": "Full image",
        "Output": "Line polygons"
    },
    "OCR Recognition": {
        "Model": "TrOCR (microsoft/trocr-base-handwritten)",
        "Purpose": "Handwritten text recognition",
        "Input": "Line crops (original resolution)",
        "Output": "Text + confidence"
    },
    "Embeddings": {
        "Model": "Sentence-Transformers MiniLM",
        "Purpose": "Semantic retrieval for RAG",
        "Input": "Text queries and topics",
        "Output": "Dense embeddings"
    },
    "Reconstruction": {
        "Model": "FLAN-T5 (google/flan-t5-large)",
        "Purpose": "OCR noise recovery",
        "Input": "Noisy text + context",
        "Output": "Cleaned text (preserving errors)"
    }
}

for component, info in models_info.items():
    print(f"\n{component}:")
    for key, val in info.items():
        print(f"  {key}: {val}")

print("\n" + "="*80)
print("APPROXIMATE VRAM REQUIREMENTS")
print("="*80)
print("\nBase Configuration (TrOCR-base + FLAN-T5-large):")
print("  - Segmentation: ~100MB")
print("  - TrOCR: ~500MB")
print("  - Embeddings: ~200MB")
print("  - FLAN-T5: ~6-7GB")
print("  - Runtime: ~8-9GB total")
print("  → Recommended GPU: T4 (16GB) or better")

print("\nLarge Configuration (TrOCR-large + FLAN-T5-XL):")
print("  - TrOCR: ~1GB")
print("  - FLAN-T5-XL: ~10-11GB")
print("  - Runtime: ~12-13GB total")
print("  → Recommended GPU: A100 (40GB+)")

print("\n" + "="*80)

## Cell 20: Citation & License

In [ ]:
print("="*80)
print("RESEARCH-GRADE HANDWRITTEN OCR RECONSTRUCTION PIPELINE")
print("="*80)

print("""
DESIGN PRINCIPLES:

✓ Modular Architecture
  - Separate stages: segmentation → OCR → confidence → RAG → reconstruction
  - Each stage independently testable and replaceable
  - Clean interfaces between components

✓ Confidence-Aware Processing
  - Word-level confidence scores from OCR
  - Selective reconstruction of low-confidence regions
  - Configurable confidence thresholds

✓ Student Error Preservation (CRITICAL)
  - Does NOT correct grammar
  - Does NOT improve answers
  - Does NOT add missing information
  - Only recovers OCR noise/corruption
  - Preserves student's original intent

✓ Open-Source RAG
  - No paid APIs (OpenAI, etc.)
  - Sentence-transformers for embeddings
  - Expandable topic database
  - Future: FAISS, ChromaDB, Qdrant support

✓ Production-Quality Code
  - Type hints and dataclasses
  - Logging and profiling
  - Error handling
  - Async-ready architecture

✓ Google Colab Optimized
  - FP16 for memory efficiency
  - GPU batching
  - Progressive model loading
  - VRAM monitoring
""")

print("="*80)
print("ACKNOWLEDGMENTS")
print("="*80)
print("""
Models & Libraries:
  - Kraken: https://github.com/mittagessen/kraken
  - TrOCR: Microsoft Research
  - FLAN-T5: Google Research
  - Sentence-Transformers: UKP Lab
  - PyTorch: Meta AI

Research Papers:
  - TrOCR: https://arxiv.org/abs/2109.10282
  - Sentence-Transformers: https://arxiv.org/abs/1908.10084
  - FLAN-T5: https://arxiv.org/abs/2210.11416
""")